# Day 5 (Advanced) — Production GNN Pipeline: Deep Dive into gnnredox/

**Duration:** 3–4 hours (self-paced, optional)
**Audience:** Bootcamp graduates who completed days 1–5 and want to explore production research code
**Goal:** Understand and run the full gnnredox/ pipeline with real tmQM data and multiple GNN architectures.

---

## Learning objectives

- Load and preprocess real tmQM dataset (2,267 iron(II) complexes) with quality filters.
- Understand the difference between bootcamp features (toy) and research features (charges, SOAP, 3D coords).
- Compare multiple GNN architectures: GCN, GAT, DimeNet++, SchNet.
- Implement rigorous validation: k-fold stratified cross-validation instead of single train/val/test split.
- Manage advanced training: custom LR scheduling, early stopping, hyperparameter sweeps.

---

## Before you start

This elective builds directly on the "Bridge to Advanced Research" section in [`day5.md`](./day5.md) (Part 6), which gives a high-level bootcamp-vs-production differences table and the environment setup steps. Read that first if you have not already. This document goes deeper into the code and hands-on exercises.

All code cells assume the production repository is available as a local clone named `gnnredox/` (upstream: https://github.com/alvarovm/Fe-Redox-GNN). Start the kernel from the workspace root, set `GNNREDOX_PATH`, or adjust `REPO_PATH` if your clone lives elsewhere.

---

## Part 1 — Data loading and real-world preprocessing (45 min)

### 1.1 The tmQM dataset

The **tmQM database** contains ~2,267 iron(II) complexes with:
- **XYZ structures** (3D atomic coordinates)
- **Reduction potentials** (experimental redox property we predict)
- **Formal charges** (computed from quantum chemistry)
- **Bond orders** (connectivity and bond type information)

After quality filtering (removing invalid valences, inconsistent ligands, Fe formal charge < 0 or > 3), we retain ~1,900–2,100 usable complexes.

### 1.2 Data loading and filtering pipeline



In [ ]:
import os
import pandas as pd
import pickle
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

# Load the full dataset and the available desolvated structures.
from pathlib import Path

configured_path = os.environ.get('GNNREDOX_PATH')
candidates = [
    Path(configured_path).expanduser() if configured_path else None,
    Path.cwd() / 'gnnredox',
    Path.cwd().parent / 'gnnredox',
    Path.cwd().parent.parent / 'gnnredox',
]
REPO_PATH = next((path for path in candidates if path and path.is_dir()), None)
if REPO_PATH is None:
    raise FileNotFoundError('Set GNNREDOX_PATH to your local gnnredox clone.')

full_data_df = pd.read_csv(REPO_PATH / 'Data' / 'tmqm_redox_data_full_data.csv')
print(f'Full dataset: {len(full_data_df)} complexes')

with open(REPO_PATH / 'Data' / 'tmc_frm_xyz2mol_desolvated_all_df.pkl', 'rb') as f:
    desolvated_df = pickle.load(f)
desolvated_df = desolvated_df.drop_duplicates(subset='csd_code', keep='first')

print(f'Desolvated structures: {len(desolvated_df)} complexes')
print(f'Difference: {len(full_data_df) - len(desolvated_df)} complexes without desolvated structures')



### 1.3 Quality filters

The data cleaning step removes:
1. **Invalid valence**: max bond order > 7 (usually bond-determination errors)
2. **Invalid Fe charge**: formal charge < 0 or > 3 (Fe should be +2 or +3 mostly)
3. **Ligand mismatch**: desolvated and solvated structures have different ligand counts (geometry changed too much)



In [ ]:
# Example: filter out bad valences and charges
rows_to_delete = []
for idx, row in desolvated_df.iterrows():
    mol = row['mol']
    atoms = mol.GetAtoms()
    max_valence = max(atom.GetDegree() for atom in atoms)
    fe_charge = None
    for atom in atoms:
        if atom.GetSymbol() == 'Fe':
            fe_charge = atom.GetFormalCharge()

    if max_valence > 7 or fe_charge is None or not 0 <= fe_charge <= 3:
        rows_to_delete.append(idx)

desolvated_df = desolvated_df.drop(rows_to_delete).reset_index(drop=True)
print(f'After filtering: {len(desolvated_df)} complexes remain')



### 1.4 Run the full analysis

Open `gnnredox/Data_analysis.ipynb` from the gnnredox repository to see:
- Redox potential distribution (histogram)
- PCA of Morgan fingerprints
- TSNE visualization
- Ligand classification and trends

---

## Part 2 — Advanced feature engineering (60 min)

### 2.1 From bootcamp to research features

**Bootcamp features** (day 4):
- One-hot element encoding (length = number of unique elements)
- Atomic number $Z$ (normalized by 100)
- Electronegativity (normalized by 4)
- Covalent radius (normalized by 2)
- 3D position (centered, normalized by 10)

**Research features** (gnnredox):
- All bootcamp features, plus:
- **Formal charge** (0, +1, +2, etc.)
- **SOAP descriptors** — local atomic environment vectors (local density, neighbors)
- **Morgan fingerprints** — molecular-level topological/chemical features
- **3D coordinates** — essential for models like DimeNet and SchNet that use angles and distances

### 2.2 Why 3D coordinates matter

The bootcamp GCN uses coordinates as node features but doesn't explicitly compute geometric properties. Advanced models use:
- **DimeNet++**: angles and distances between atoms
- **SchNet**: distance-dependent filters with cutoff (8 Å)

These models are more chemically accurate because they respect 3D molecular geometry.

### 2.3 Feature computation example



In [ ]:
from dscribe.descriptors import SOAP
from sklearn.decomposition import PCA
import numpy as np

# SOAP: Smooth Overlap of Atomic Positions. Include every element in the real structures.
species = sorted({symbol for atoms in desolvated_df['desolv_atoms'] for symbol in atoms.get_chemical_symbols()})
soap = SOAP(
    species=species,
    r_cut=5.0,  # Cutoff radius in Angstroms
    n_max=8,    # Radial basis functions
    l_max=6     # Angular basis functions
)

# Compute one mean SOAP vector per molecule.
soap_features = []
for _, row in desolvated_df.iterrows():
    soap_vec = soap.create(row['desolv_atoms'])
    soap_features.append(soap_vec.mean(axis=0))

soap_features = np.array(soap_features)

# Reduce dimensionality
pca = PCA(n_components=min(50, *soap_features.shape))
soap_pca = pca.fit_transform(soap_features)

print(f'SOAP variance explained (50 components): {pca.explained_variance_ratio_.sum():.2%}')



### 2.4 Feature importance from Data_analysis.ipynb

Run the notebook to see:
- Which ligands dominate the dataset?
- Do redox potentials correlate with coordination number?
- Are there distinct "families" of complexes?

---

## Part 3 — Multiple model architectures (60 min)

### 3.1 Recap: bootcamp GCN

```
Linear(node_features -> 64)
GCNConv(64 -> 64) + BatchNorm + ReLU + Residual
GCNConv(64 -> 64) + BatchNorm + ReLU + Residual
GCNConv(64 -> 64) + BatchNorm + ReLU + Residual
global_mean_pool(64)
Linear(64 -> 64) + ReLU
Linear(64 -> 32) + ReLU
Linear(32 -> 1)  [predict redox potential]
```

**Bootcamp hyperparameters**:
- Hidden dim: **64**
- Batch size: **32**
- Learning rate: **1e-3**
- Epochs: **150**
- Early stopping patience: **20**

### 3.2 Production GCN (gnnredox)

Same architecture but scaled up:

```
Linear(node_features -> 512)
GCNConv(512 -> 512) + BatchNorm + ReLU
GCNConv(512 -> 512) + BatchNorm + ReLU
GCNConv(512 -> 512) + BatchNorm + ReLU
global_mean_pool(512)
Linear(512 -> 256) + ReLU
Linear(256 -> 128) + ReLU
Linear(128 -> 1)  [predict redox potential]
```

**Production hyperparameters**:
- Hidden dim: **512** (8x bootcamp)
- Batch size: **128** (4x bootcamp)
- Learning rate schedule: **1e-4 -> 1e-3 -> 1e-4** (custom, not single LR)
- Epochs: **250+** (with early stopping)
- Early stopping patience: **50** (2.5x bootcamp)

### 3.3 Graph Attention Networks (GAT)

Instead of averaging neighbor embeddings (like GCN), GAT learns **attention weights** per edge:

$$\alpha_{ij} = \frac{\exp(a^T[\mathbf{h}_i \| \mathbf{h}_j])}{\sum_{k \in \mathcal{N}(i)} \exp(a^T[\mathbf{h}_i \| \mathbf{h}_k])}$$

Then aggregates:

$$\mathbf{h}_i^{(l+1)} = \sigma\left(\sum_{j \in \mathcal{N}(i) \cup \{i\}} \alpha_{ij} W^{(l)} \mathbf{h}_j^{(l)}\right)$$

**Multi-head attention**: run 3–4 independent attention heads, concatenate results.

**When to use GAT vs GCN**:
- GCN: faster, good for homogeneous graphs, fewer parameters
- GAT: slower, better when different neighbors have different importance

### 3.4 DimeNet++ (3D-aware)

Directional Graph Neural Networks that use **angles** and **distances** directly:

$$\phi(d_{ij}, d_{jk}, \theta_{ijk}) = \text{basis expansion using RBF + spherical harmonics}$$

**Key insight**: doesn't construct a fixed graph; instead uses all atom pairs within a cutoff (8 Å) and refines connections based on 3D geometry.

**When to use**:
- When 3D structure is crucial
- When you have precise atomic coordinates
- For molecular property prediction (often better than GCN/GAT)

### 3.5 SchNet (continuous filters)

Uses distance-dependent continuous filters instead of fixed convolutions:

$$\mathbf{h}_i^{(l+1)} = \sum_{j: d_{ij} < \text{cutoff}} \text{filter}(d_{ij}) \odot \mathbf{h}_j^{(l)}$$

where $\text{filter}(d)$ is learned as a continuous function of distance via Gaussian basis expansion.

**When to use**:
- When you want smooth distance-dependent interactions
- Cheaper than DimeNet
- Good for properties sensitive to atomic distances

### 3.6 Comparison table

| Model | Complexity | Memory | Speed | Best for |
|---|---|---|---|---|
| **GCN** | Low | Low | Fast | Prototyping, large graphs |
| **GAT** | Medium | Medium | Medium | Heterogeneous importance, interpretability |
| **DimeNet++** | High | High | Slow | 3D geometry-critical properties |
| **SchNet** | Medium | Medium | Medium | Distance-dependent interactions |

The corresponding production notebooks in `gnnredox/Model_training/` are:
- `1b_GCN_base.ipynb`, `1_GCN_model_desolvated-structrs.ipynb` (GCN)
- `2_GAT_model_desolvated_structrs.ipynb` (GAT)
- `3_Dimenet_model_desolvated-structrs.ipynb` (DimeNet++)
- `4_Schnet_model_desolvated-structrs.ipynb` (SchNet)

---

## Part 4 — Cross-validation and rigorous evaluation (45 min)

### 4.1 Why k-fold CV vs single split?

**Bootcamp** (single split):
- Train: 350 samples -> Val: 75 samples -> Test: 75 samples
- Single train/val/test gives one estimate of performance
- **Problem**: high variance (depends on which samples went where)

**Research** (k-fold CV):
- Partition data into $k$ folds (e.g., $k=5$)
- Train $k$ models: fold 1–4 train, fold 5 test; then fold 1–3,5 train, fold 4 test; etc.
- Average performance across $k$ folds
- **Benefit**: more robust estimate, use all data for training and testing

### 4.2 Stratified CV for imbalanced data

Redox potentials are **not uniformly distributed** — some values are rare. **Stratified CV** ensures each fold has similar redox potential distributions (no fold is accidentally biased toward high or low values).



```python
# Illustrative: adapt this pattern inside a production training notebook.
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Discretize redox potentials into bins for stratification
y_binned = pd.cut(redox_potentials, bins=10, labels=False)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y_binned)):
    train_data = dataset[train_idx]
    test_data = dataset[test_idx]
    print(f'Fold {fold_idx}: train {len(train_idx)}, test {len(test_idx)}')
```


### 4.3 Custom LR scheduling

**Bootcamp**: ReduceLROnPlateau (reduces LR if val loss plateaus)

**Research**: custom schedule



In [ ]:
def lr_schedule(epoch):
    if epoch < 50:
        return 1e-4  # Warm-up
    elif epoch < 150:
        return 1e-3  # Main training
    else:
        return 1e-4  # Fine-tuning



**Why**: gives the optimizer more control; helps escape local minima during warmup and fine-tuning.

### 4.4 Early stopping with patience=50

Wait 50 epochs without improvement before stopping (vs bootcamp's patience=20).

**Trade-off**:
- Larger patience: may train longer but find better minimum
- Smaller patience: stop sooner, avoid overfitting but may undershoot

For real data with ~2,000 samples and rigorous CV, patience=50 is reasonable.

### 4.5 Run 1b_GCN_base.ipynb with 3-fold CV

Open `gnnredox/Model_training/1b_GCN_base.ipynb` and modify to use 3-fold CV:



```python
# Illustrative: adapt this pattern inside 1b_GCN_base.ipynb; do not run standalone.
from sklearn.model_selection import StratifiedKFold

fold_results = []
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(data, y_binned)):
    print(f'\nTraining fold {fold_idx + 1}/3')
    train_loader = DataLoader([data[i] for i in train_idx], batch_size=128, shuffle=True)
    test_loader = DataLoader([data[i] for i in test_idx], batch_size=128)

    model = GCN(...).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

    for epoch in range(250):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_rmse = evaluate(model, test_loader, device)
        scheduler.step()
        # Early stopping logic

    final_rmse = evaluate(model, test_loader, device)['rmse']
    fold_results.append(final_rmse)

print(f'Mean test RMSE: {np.mean(fold_results):.4f} +/- {np.std(fold_results):.4f}')
```


---

## Part 5 — Reproducibility and model persistence (30 min)

### 5.1 Checkpoint management

Save best model per fold:



```python
# Illustrative: insert into a training loop after defining val_rmse and best_val_rmse.
if val_rmse < best_val_rmse:
    best_val_rmse = val_rmse
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': {'hidden_dim': 512, 'num_layers': 3},
        'fold': fold_idx,
        'epoch': epoch,
        'val_rmse': val_rmse
    }, f'gnn_fold_{fold_idx}_best.pt')
```


### 5.2 Hyperparameter sweeps

Grid search over hidden dimensions and number of layers:



```python
# Illustrative: requires the GCN class and training/evaluation functions from the production notebook.
import itertools

param_grid = {
    'hidden_dim': [64, 128, 256, 512],
    'num_gnn_layers': [2, 3, 4],
    'dropout': [0.0, 0.1, 0.2]
}

best_config = None
best_mean_rmse = float('inf')

for config in itertools.product(*param_grid.values()):
    hidden_dim, num_layers, dropout = config
    fold_rmses = []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(data, y_binned)):
        model = GCN(in_dim, hidden_dim, num_layers, dropout).to(device)
        # ... training loop ...
        fold_rmses.append(final_rmse)

    mean_rmse = np.mean(fold_rmses)
    if mean_rmse < best_mean_rmse:
        best_mean_rmse = mean_rmse
        best_config = config

print(f'Best config: hidden_dim={best_config[0]}, num_layers={best_config[1]}, dropout={best_config[2]}')
print(f'Mean RMSE: {best_mean_rmse:.4f}')
```


### 5.3 Ensemble predictions

Combine predictions from all folds:



```python
# Illustrative: requires saved fold checkpoints and the production model definition.
fold_predictions = []

for fold_idx in range(3):
    ckpt = torch.load(f'gnn_fold_{fold_idx}_best.pt')
    model = GCN(...).to(device)
    model.load_state_dict(ckpt['model_state_dict'])

    preds = []
    for batch in test_loader:
        batch = batch.to(device)
        with torch.no_grad():
            out = model(batch)
        preds.append(out.detach().cpu().numpy())
    fold_predictions.append(np.concatenate(preds))

ensemble_preds = np.mean(fold_predictions, axis=0)
ensemble_rmse = rmse(y_test, ensemble_preds)
print(f'Ensemble RMSE: {ensemble_rmse:.4f}')
```


### 5.4 Save for deployment

Final model package with everything needed for inference:



```python
# Illustrative: run after selecting best_model, best_config, and scaler.
from datetime import datetime

torch.save({
    'model_state_dict': best_model.state_dict(),
    'config': best_config,
    'fold_results': fold_results,
    'ensemble_rmse': ensemble_rmse,
    'feature_scaler': scaler,  # e.g., StandardScaler
    'training_date': datetime.now().isoformat()
}, 'gnn_production_model.pt')
```


---

## Exercises

### Exercise A.1 — Explore the real tmQM dataset

1. Open `gnnredox/Data_analysis.ipynb`.
2. Run the data loading and filtering pipeline.
3. Report:
   - How many complexes start in the dataset?
   - How many pass the quality filters?
   - What percentage were removed?
4. Examine the redox potential distribution (histogram):
   - What is the mean and std?
   - Any outliers?
5. Run the PCA on Morgan fingerprints:
   - How many components needed for 90% variance?
   - Compare to bootcamp PCA results (which used different features).

### Exercise A.2 — Train a production GCN with 3-fold CV

1. Open `gnnredox/Model_training/1b_GCN_base.ipynb`.
2. Modify the training script to use 3-fold stratified CV (see Part 4 code above).
3. Use real desolvated tmQM data (not synthetic).
4. Train with:
   - Hidden dim: 512 (vs bootcamp's 64)
   - Batch size: 128 (vs bootcamp's 32)
   - Custom LR schedule: 1e-4 -> 1e-3 -> 1e-4
   - Early stopping patience: 50
5. Report:
   - Mean test RMSE across folds (with std)?
   - Best fold RMSE? Worst fold RMSE?
   - Why do you think folds differ?
   - How does it compare to your Day 4 bootcamp GNN RMSE?

**Hint**: The production model will likely perform better because it has more capacity (512 hidden) and more training data (2,000+ vs 350 samples).

### Exercise A.3 — Enhance features and retrain (optional, open-ended)

1. Add **formal charge** as a node feature (look at the RDKit Mol objects in desolvated_df).
2. Optionally add **SOAP descriptors** (see Part 2.3 example).
3. Retrain the GCN from Exercise A.2 with the new features.
4. Compare test RMSE:
   - Baseline (Exercise A.2): [your number]
   - With charges: [retrain and report]
   - With charges + SOAP: [retrain and report]
5. Which feature improved performance the most?
6. Did any feature hurt performance? Why might that be?

**Hints**:
- Formal charge is a single scalar per atom (easy to add).
- SOAP is 50-100 dimensional per atom (requires dimensionality reduction).
- Not all features help; sometimes more features = overfitting on small validation folds.

---

**Mentor checkpoint A.1**

- Exercise A.1 completed: data filters understood, variance analysis done.
- Exercise A.2 completed: 3-fold CV trained, RMSE reported, fold variability discussed.
- Exercise A.3 (optional): feature engineering attempted and performance compared.
- Proceed only after confirmation.

---

## Summary

| Component | Bootcamp (day 4) | Advanced (gnnredox) |
|---|---|---|
| **Data** | 500 synthetic, 70/15/15 split | 2,267 real tmQM, 3–5 fold CV |
| **Features** | Simple atom features | Charges, SOAP, Morgan FP, 3D coords |
| **Models** | 1 (GCN-64) | 4 (GCN-512, GAT-512, DimeNet, SchNet) |
| **Training** | Single LR, patience=20 | Custom schedule, patience=50 |
| **Validation** | Single train/val/test | Stratified k-fold CV |
| **Time commitment** | 4 hours (bootcamp) | 3–4 hours (self-paced) |

---

## After the advanced workshop

You have now seen the full spectrum: from pedagogical toy models (bootcamp) to production research pipelines (gnnredox).

**Next steps if you want to continue**:

1. **Contribute to gnnredox**: add a new model, improve preprocessing, write docs.
2. **Try other datasets**: Materials Project, Open Catalyst, other chemistry benchmarks.
3. **Publish**: if your model/insights are novel, consider writing a paper.
4. **Deploy**: package your best model and expose it via an API for predictions on new iron complexes.

The skills you learned — data loading, feature engineering, model architecture, validation, reproducibility — are directly transferable to any ML research project.
